# S6 — kiểm chứng calibration val-only và evaluation bằng policy cố định

Ngày tạo: 10/10/2026. Notebook chạy trên CPU trong Conda `pdp` Python 3.10. Không cần GPU 3060/4090 cho bước này.

Kiểm chứng công thức đếm/mAP đã có, grid và NMS chéo lớp, oracle mặc định tắt, lưu policy val và nạp lại trên test bằng CLI thật. Đọc predictions bốn cases từ notebook modeling 04; metrics subset chỉ là diagnostics kỹ thuật. Không train, không cài dependencies, không tải weights hoặc tạo lại split. Input release/predictions/checkpoints giữ nguyên.

Output: `runs/evaluation/count_calibration/<run_id>/`, logs/JUnit/policy/metrics/config/gates/summary. PASS là `PASS_COUNT_CALIBRATION_ACCEPTANCE`; chưa chứng nhận calibration của model đã hội tụ hoặc pilot/Vast.ai.


In [1]:
import importlib.metadata
import json
import os
import platform
import subprocess
import sys
import uuid
import xml.etree.ElementTree as ET
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(REPO))
from autocheckout.io import load_json, md5_file, save_json
from autocheckout.model_acceptance import RunEvidence
from autocheckout.predictions import Predictions, load_predictions, save_predictions
from autocheckout.taskcfg import TaskConfig

runtime_root = REPO / 'runs/modeling/training_runtime'
runtime_override = os.environ.get('AUTOCHECKOUT_RUNTIME_SUMMARY')
RUNTIME_SUMMARY = Path(runtime_override) if runtime_override else runtime_root / load_json(runtime_root / 'latest.json')['run_id'] / 'summary.json'
ARTIFACT = REPO / 'runs/evaluation/count_calibration' / str(uuid.uuid4())
CONFIG = dict(runtime_summary=str(RUNTIME_SUMMARY), threshold_grid=[round(0.1 + 0.01*i, 2) for i in range(81)], nms_grid=[0.45], seed=0, cpu_only=True)
REQUIRED = ['environment', 'regression', 'fixture_calibration', 'locked_test_policy', 'raw_map', 'runtime_predictions', 'input_preservation']
RUN = RunEvidence(ARTIFACT, CONFIG, REQUIRED, pass_status='PASS_COUNT_CALIBRATION_ACCEPTANCE')
save_json(ARTIFACT / 'config.json', CONFIG, indent=2)
print(f"Run {ARTIFACT.name} started {RUN.data['started_at']} outputs {ARTIFACT}")

def cli(arguments, log):
    result = subprocess.run([sys.executable, '-m', *arguments], cwd=REPO, capture_output=True, text=True)
    (ARTIFACT / log).write_text(result.stdout + result.stderr)
    assert result.returncode == 0, result.stdout + result.stderr
    return result


Run af79910c-a01e-4f63-83b7-8c84ac3e49b6 started 2026-10-10T00:36:13+07:00 outputs /home/sakana/Code/AutoCheckout-CL/runs/evaluation/count_calibration/af79910c-a01e-4f63-83b7-8c84ac3e49b6


## 1. Môi trường và regression
Ghi versions, git revision/dirty, source hashes và run notebook 04 đã PASS. Chạy tests lõi đếm, predictions, calibration, mAP và runner; JUnit cần zero failures/errors/skips.


In [2]:
with RUN.gate('environment') as evidence:
    assert sys.version_info[:2] == (3, 10), sys.version
    runtime_summary = load_json(RUNTIME_SUMMARY)
    assert runtime_summary['status'] == 'PASS_PDP_CLI_RUNNER_CUDA_SMOKE'
    RUNTIME = RUNTIME_SUMMARY.parent
    runtime_config = load_json(RUNTIME / 'config.json')
    TASK_CONFIG = Path(runtime_config['task_config'])
    SUBSET = Path(runtime_config['subset_dir'])
    sources = [REPO / p for p in ['autocheckout/counting.py', 'autocheckout/predictions.py', 'tools/eval_count.py', 'tools/eval_cl.py', 'scripts/run_exp.sh', 'configs/exp/native/common.sh']]
    INPUTS = {str(p): md5_file(p) for p in [*sources, TASK_CONFIG, *SUBSET.glob('*.json')]}
    save_json(ARTIFACT / 'input_hashes.json', INPUTS, indent=2)
    evidence.update(python=sys.version, host=platform.node(), packages={p: importlib.metadata.version(p) for p in ['numpy', 'pycocotools', 'pytest']}, git_revision=subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip(), git_dirty=bool(subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True)), runtime_run_id=runtime_summary['run_id'], cpu_only=True)

with RUN.gate('regression') as evidence:
    tests = ['tests/test_counting.py', 'tests/test_predictions.py', 'tests/test_eval_count.py', 'tests/test_count_calibration.py', 'tests/test_eval_cl.py', 'tests/test_cl_metrics.py', 'tests/test_run_exp.py', 'tests/test_native_exp_configs.py']
    cli(['pytest', *tests, '-q', '--junitxml', str(ARTIFACT / 'regression.xml'), '--basetemp', str(ARTIFACT / 'regression_tmp')], 'regression.log')
    suites = list(ET.parse(ARTIFACT / 'regression.xml').getroot().iter('testsuite'))
    counts = {k: sum(int(s.get(k, 0)) for s in suites) for k in ['tests', 'failures', 'errors', 'skipped']}
    assert counts['tests'] > 0 and counts['failures'] == counts['errors'] == counts['skipped'] == 0, counts
    evidence.update(counts)
    print(counts)


{'tests': 71, 'failures': 0, 'errors': 0, 'skipped': 0}


## 2. Fixture có kết quả biết trước và policy cố định
Val có một object mỗi ảnh; object bị dự đoán bằng hai labels và một decoy score 0,20. NMS 0,45 bỏ duplicate, threshold tối ưu nhỏ nhất là 0,21. Test có một true detection score 0,15: policy val vẫn giữ 0,21, cAcc test chỉ 0,5. Sau khi lưu policy, làm paths val không còn để chứng minh evaluate không cần val; thay test GT trong fixture phải chỉ đổi metrics.


In [3]:
FIXTURE = ARTIFACT / 'fixture'
fixture_cfg = TaskConfig.from_dict(dict(name='calibration-fixture', seed=0, tasks=[dict(task_id=1, offset=0, reserved=False, classes=[dict(label=i, name=f'sku{i}', rpc_category_id=i+1) for i in range(2)])]))
fixture_cfg.save(FIXTURE / 'task_config.json')
def annotation(ids):
    return dict(images=[dict(id=i, file_name=f'{i}.jpg', width=100, height=100, level='easy') for i in ids], categories=[dict(id=i, name=f'sku{i}') for i in range(2)], annotations=[dict(id=n+1, image_id=i, category_id=0, bbox=[0, 0, 5, 5], area=25, iscrowd=0) for n, i in enumerate(ids)])
fixture_val, fixture_test = FIXTURE / 'val_full.json', FIXTURE / 'test_full.json'
save_json(fixture_val, annotation([1, 2]))
save_json(fixture_test, annotation([10, 11]))
for split, ann, ids, queries, labels, scores, boxes in [('val', fixture_val, [1, 1, 1, 2], [0, 1, 2, 0], [0, 1, 0, 0], [0.9, 0.8, 0.2, 0.9], [[0, 0, 5, 5], [0, 0, 5, 5], [20, 20, 25, 25], [0, 0, 5, 5]]), ('test', fixture_test, [10, 11], [0, 0], [0, 0], [0.15, 0.9], [[0, 0, 5, 5]]*2)]:
    save_predictions(FIXTURE / f'task_1/pred_{split}.npz', Predictions(ids, queries, labels, scores, boxes, meta=dict(task_id=1, seen_classes=2, split=split, ann_md5=md5_file(ann), coordinates='native_pixels')))
fixture_policy = FIXTURE / 'calibration_count.json'
fixture_common = ['tools.eval_count', '--run-dir', str(FIXTURE), '--task-config', str(FIXTURE / 'task_config.json')]
with RUN.gate('fixture_calibration') as evidence:
    cli([*fixture_common, '--calibrate-only', '--val-ann', str(fixture_val), '--nms-grid', '0,0.45', '--policy-out', str(fixture_policy)], 'fixture_calibration.log')
    policy = load_json(fixture_policy)
    selected = policy['stages']['1']
    assert selected['score_threshold'] == 0.21 and selected['nms_iou'] == 0.45
    assert selected['val_scores']['cAcc'] == 1 and not (FIXTURE / 'metrics_count_test.json').exists()
    evidence.update(selected=selected, test_not_read=True)

with RUN.gate('locked_test_policy') as evidence:
    policy_hash = md5_file(fixture_policy)
    fixture_val.rename(FIXTURE / 'val_full.unavailable.json')
    (FIXTURE / 'task_1/pred_val.npz').rename(FIXTURE / 'task_1/pred_val.unavailable.npz')
    cli([*fixture_common, '--policy-in', str(fixture_policy), '--test-ann', str(fixture_test)], 'fixture_test.log')
    metrics = load_json(FIXTURE / 'metrics_count_test.json')
    assert metrics['stages']['1']['threshold']['value'] == 0.21
    assert metrics['stages']['1']['test']['overall']['cAcc'] == 0.5
    assert not metrics['oracle_enabled'] and 'oracle' not in metrics['stages']['1']
    changed = load_json(fixture_test)
    changed['annotations'].append(dict(id=3, image_id=11, category_id=0, bbox=[20, 20, 5, 5], area=25, iscrowd=0))
    changed_ann = FIXTURE / 'test_changed.json'
    save_json(changed_ann, changed)
    preds = load_predictions(FIXTURE / 'task_1/pred_test.npz')
    preds.meta['ann_md5'] = md5_file(changed_ann)
    save_predictions(FIXTURE / 'task_1/pred_test.npz', preds)
    cli([*fixture_common, '--policy-in', str(fixture_policy), '--test-ann', str(changed_ann)], 'fixture_test_changed.log')
    changed_metrics = load_json(FIXTURE / 'metrics_count_test.json')
    assert changed_metrics['stages']['1']['test']['overall']['cAcc'] == 0
    assert md5_file(fixture_policy) == policy_hash
    evidence.update(threshold=0.21, initial_test_cAcc=0.5, changed_test_cAcc=0.0, policy_unchanged=True, val_unavailable=True, oracle_enabled=False)

with RUN.gate('raw_map') as evidence:
    raw_hash = md5_file(FIXTURE / 'task_1/pred_test.npz')
    cli(['tools.eval_cl', '--run-dir', str(FIXTURE), '--split', 'test', '--ann', str(changed_ann), '--task-config', str(FIXTURE / 'task_config.json')], 'fixture_map.log')
    assert md5_file(FIXTURE / 'task_1/pred_test.npz') == raw_hash
    evidence.update(raw_predictions_unchanged=True, report=str(FIXTURE / 'metrics_cl_test.json'))


## 3. Predictions thật từ notebook 04
Bốn cases ResNet/ConvNeXt × 640/800 dùng predictions/checkpoint qua symlink read-only về mặt luồng xử lý; metrics/policy ghi trong artifact mới. Không copy checkpoint. Mỗi case có tiến trình calibration val-only, tiến trình evaluate test dùng policy và tiến trình mAP đọc output detector thô. Hash input phải giữ nguyên; các chỉ số subset không phải kết quả nghiên cứu.


In [4]:
with RUN.gate('runtime_predictions') as evidence:
    reports = []
    for experiment in ['EXP-B1', 'EXP-B2']:
        for resolution in [640, 800]:
            name = f'{experiment}_{resolution}'
            source = RUNTIME / name / experiment / 'task_1'
            assert source.is_dir(), source
            for filename in ['pred_val.npz', 'pred_test.npz', 'task_final.pth']:
                p = source / filename
                INPUTS[str(p)] = md5_file(p)
            case = ARTIFACT / name
            case.mkdir()
            (case / 'task_1').symlink_to(source, target_is_directory=True)
            policy_path = case / 'calibration_count.json'
            common = ['tools.eval_count', '--run-dir', str(case), '--task-config', str(TASK_CONFIG)]
            cli([*common, '--calibrate-only', '--val-ann', str(SUBSET / 'val_full.json'), '--nms-grid', '0.45', '--policy-out', str(policy_path)], f'{name}_calibration.log')
            policy_hash = md5_file(policy_path)
            cli([*common, '--policy-in', str(policy_path), '--test-ann', str(SUBSET / 'test_full.json')], f'{name}_counting.log')
            result = load_json(case / 'metrics_count_test.json')
            assert not result['oracle_enabled'] and 'oracle' not in result['stages']['1']
            assert result['calibration_policy_md5'] == policy_hash == md5_file(policy_path)
            cli(['tools.eval_cl', '--run-dir', str(case), '--split', 'test', '--ann', str(SUBSET / 'test_full.json'), '--task-config', str(TASK_CONFIG)], f'{name}_map.log')
            selected = load_json(policy_path)['stages']['1']
            assert selected['checkpoint_md5'] == INPUTS[str(source / 'task_final.pth')]
            reports.append(dict(case=name, status='PASS', threshold=selected['score_threshold'], nms_iou=selected['nms_iou'], diagnostic_counting=result['stages']['1']['test']['overall']))
    evidence.update(cases=reports, metrics_are_diagnostics=True, gpu_used=False)
    print(json.dumps(reports, indent=2))

with RUN.gate('input_preservation') as evidence:
    after = {p: md5_file(p) for p in INPUTS}
    assert after == INPUTS
    save_json(ARTIFACT / 'input_hashes.json', INPUTS, indent=2)
    evidence.update(verified=True, files=len(INPUTS), source_release_predictions_checkpoints_unchanged=True)


[
  {
    "case": "EXP-B1_640",
    "status": "PASS",
    "threshold": 0.1,
    "nms_iou": 0.45,
    "diagnostic_counting": {
      "cAcc": 0.0,
      "ACD": 4.0,
      "mCCD": 1.0,
      "mCIoU": 0.0,
      "K": 100,
      "K_eff": 2
    }
  },
  {
    "case": "EXP-B1_800",
    "status": "PASS",
    "threshold": 0.1,
    "nms_iou": 0.45,
    "diagnostic_counting": {
      "cAcc": 0.0,
      "ACD": 4.0,
      "mCCD": 1.0,
      "mCIoU": 0.0,
      "K": 100,
      "K_eff": 2
    }
  },
  {
    "case": "EXP-B2_640",
    "status": "PASS",
    "threshold": 0.1,
    "nms_iou": 0.45,
    "diagnostic_counting": {
      "cAcc": 0.0,
      "ACD": 4.0,
      "mCCD": 1.0,
      "mCIoU": 0.0,
      "K": 100,
      "K_eff": 2
    }
  },
  {
    "case": "EXP-B2_800",
    "status": "PASS",
    "threshold": 0.1,
    "nms_iou": 0.45,
    "diagnostic_counting": {
      "cAcc": 0.0,
      "ACD": 4.0,
      "mCCD": 1.0,
      "mCIoU": 0.0,
      "K": 100,
      "K_eff": 2
    }
  }
]


## 4. Kết luận
PASS cần đủ 7 gates. Calibration kỹ thuật đã được kiểm chứng; policy sản phẩm/nghiên cứu phải sinh từ predictions val của từng checkpoint pilot thật, không tái sử dụng policy subset này. Tiếp theo: môi trường GPU training/native kernel hoặc fallback có đo throughput → pilot EXP-B1/B2 cùng budget → full teacher Task 1→2 smoke → protocol 5 task.


In [5]:
RUN.data.update(gpu_used=False, pilot_calibrated=False, full_training_ready=False, limitations=['Fixture and Task 1 subset only; no research metrics or converged-model calibration.', 'Training GPU/native kernel/throughput and pilot remain separate gates.'])
summary = RUN.finish()
assert summary['status'] == 'PASS_COUNT_CALIBRATION_ACCEPTANCE', summary
save_json(ARTIFACT.parent / 'latest.json', dict(run_id=ARTIFACT.name, status=summary['status'], finished_at=summary['finished_at']), indent=2)
print(json.dumps(dict(run_id=ARTIFACT.name, started_at=summary['started_at'], finished_at=summary['finished_at'], status=summary['status'], gates=len(summary['gates']), regression=summary['gates']['regression']), indent=2))


{
  "run_id": "af79910c-a01e-4f63-83b7-8c84ac3e49b6",
  "started_at": "2026-10-10T00:36:13+07:00",
  "finished_at": "2026-10-10T00:36:24+07:00",
  "status": "PASS_COUNT_CALIBRATION_ACCEPTANCE",
  "gates": 7,
  "regression": {
    "status": "PASS",
    "started_at": "2026-10-10T00:36:13+07:00",
    "tests": 71,
    "failures": 0,
    "errors": 0,
    "skipped": 0,
    "finished_at": "2026-10-10T00:36:16+07:00"
  }
}
